<a href="https://colab.research.google.com/github/br-raghav/STOCK-MARKET-PERFORMANCE-PREDICTION-USING-LSTM/blob/main/STOCK_MARKET_PERFORMANCE_PREDICTION_USING_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# STOCK MARKET PERFORMANCE PREDICTION USING LSTM
# ============================================================

# This project uses historical stock market data to predict
# future stock prices using a Long Short-Term Memory (LSTM)
# neural network.
#
# Example:
# The project uses Apple Inc. (AAPL) stock data obtained
# from the Tiingo API.
#
# The project then:
#
# 1. Connects to the Tiingo API and collects historical
#    AAPL stock price data.
#
# 2. Loads the stock data into a Pandas DataFrame.
#
# 3. Selects the closing price because it is the main value
#    used for predicting future stock prices.
#
# 4. Scales the closing-price data between 0 and 1 so that
#    it can be processed more effectively by the LSTM model.
#
# 5. Splits the data into training and testing datasets.
#
# 6. Creates sequences of previous stock prices that are
#    used to predict the next price.
#
# 7. Builds a stacked LSTM neural network using TensorFlow
#    and Keras.
#
# 8. Trains the LSTM model using historical stock prices.
#
# 9. Uses the trained model to predict prices for the
#    training and testing datasets.
#
# 10. Calculates RMSE to measure the prediction error.
#
# 11. Uses the latest available prices to predict the next
#     30 days of stock prices.
#
# The project demonstrates how a time-series machine learning
# model can be applied to historical financial data.
# ============================================================



In [ ]:
### checking the jupyter notebook environment
# checking and displaying the pandas

import pandas as pd
import pandas_datareader

print("pandas:", pd.__version__)
print("pandas_datareader:", pandas_datareader.__version__)
# checking the tiingo integration with jupyter notebook

# Import Tiingo reader from pandas_datareader
import pandas as pd
import pandas_datareader.data as pdr

# Tiingo API key
api_key = "YOUR_API_KEY"

# Fetch AAPL stock data from Tiingo
df = pdr.get_data_tiingo("AAPL", api_key=api_key)
### displaying the stock price dataframe
# display first 5 rows of data
df.head()
# display last 5 rows
df.tail()
# shows the shape (rows, columns)
df.shape
# shows info about columns, data types, missing values
df.info()
### impoting the libraries
# Numpy for numerical operations
import numpy as np

# Pandas for data handling
import pandas as pd

# Matplotlib for plotting
import matplotlib.pyplot as plt

# Math module for RMSE calculation
import math

# Scikit-learn for scaling data and computing RMSE
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

# TensorFlow Keras for building LSTM model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
### selecting and ploting the Closing Price
# Extract only the 'close' column for prediction
close_price = df['close']

# Show first 5 closing prices
close_price.head()
# ploting the closing price over time
close_price = df['close'].astype(float)

plt.figure(figsize=(10,5))
plt.plot(close_price.values)
plt.title("Apple Stock Closing Price")
plt.xlabel("Time")
plt.ylabel("Price")
plt.show()
### scaling the Data
# scaling data to range 0-1 for LSTM
scaler = MinMaxScaler(feature_range=(0,1))

# reshaping data to 2D array (required by scaler)
close_scaled = scaler.fit_transform(np.array(close_price).reshape(-1,1))
### spliting the data for training and testing
# 65% for training
train_size = int(len(close_scaled) * 0.65)

# remaining 35% for testing
test_size = len(close_scaled) - train_size

# spliting the data
train_data = close_scaled[:train_size]
test_data = close_scaled[train_size:]

# prints the sizes
train_size, test_size
### creating the Dataset Function
# function to convert series to supervised learning format
def create_dataset(data, step):
    X, y = [], []
    # Loop over data
    for i in range(len(data) - step - 1):
        # X = previous 'step' values
        X.append(data[i:(i+step), 0])
        # y = next value
        y.append(data[i + step, 0])
    return np.array(X), np.array(y)
### preparing Training and Testing Data
# number of time steps to look back
time_steps = 100

# prepares training dataset
X_train, y_train = create_dataset(train_data, time_steps)

# prepares testing dataset
X_test, y_test = create_dataset(test_data, time_steps)

# prints shapes
X_train.shape, y_train.shape
### reshaping Data for LSTM
# LSTM expects input in 3D: (samples, time_steps, features)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test  = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
### building LSTM Model
# initialize Sequential model
model = Sequential()

# first LSTM layer with 50 units, return sequences for stacking
model.add(LSTM(50, return_sequences=True, input_shape=(time_steps, 1)))

# second LSTM layer with 50 units
model.add(LSTM(50, return_sequences=True))

# third LSTM layer with 50 units
model.add(LSTM(50))

# output layer with 1 neuron (predicted price)
model.add(Dense(1))

# compiling the model using mean squared error loss and Adam optimizer
model.compile(loss='mean_squared_error', optimizer='adam')

# shows model summary
model.summary()
### training the Model
# fiting the model on training data, validate on test data
model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=100,       # Number of epochs
    batch_size=64,    # Batch size
    verbose=1
)
### prediction maker
# predicting on training data
train_pred = model.predict(X_train)

# predictin on test data
test_pred  = model.predict(X_test)
### inverse scaling
# transforming predictions back to original scale
train_pred = scaler.inverse_transform(train_pred)
test_pred  = scaler.inverse_transform(test_pred)
### RMSE evaluation
# calculating Root Mean Squared Error for train data
train_rmse = math.sqrt(mean_squared_error(y_train, train_pred))

# calculating RMSE for test data
test_rmse  = math.sqrt(mean_squared_error(y_test, test_pred))

# displaying RMSE
train_rmse, test_rmse
### plot Predictions vs Actual data
# preparing empty array for train plot
train_plot = np.empty_like(close_scaled)
train_plot[:] = np.nan
train_plot[time_steps:len(train_pred)+time_steps] = train_pred

# preparing empty array for test plot
test_plot = np.empty_like(close_scaled)
test_plot[:] = np.nan
test_plot[len(train_pred)+(time_steps*2)+1:len(close_scaled)-1] = test_pred


# ploting predicted vs actual
plt.figure(figsize=(10,5))
plt.plot(scaler.inverse_transform(close_scaled), label="Actual Price")
plt.plot(train_plot, label="Train Prediction")
plt.plot(test_plot, label="Test Prediction")
plt.legend()
plt.show()
### predicting next 30 days
# number of future days to predict
future_days = 30

# list to store future predictions
future_output = []

# convert last 100 points to list
temp_input = last_100.flatten().tolist()

# a for loop to predict future days
for i in range(future_days):
    x_input = np.array(temp_input[-time_steps:], dtype=np.float32).reshape(1, time_steps, 1)
    yhat = model.predict(x_input, verbose=0)

    predicted_value = float(yhat[0][0])
    temp_input.append(predicted_value)
    future_output.append(predicted_value)
### ploting Future Forecast
# reshape future_output for inverse scaling
future_output = np.array(future_output).reshape(-1, 1)
future_output = scaler.inverse_transform(future_output)

plt.figure(figsize=(10,5))

# Historical last 200 points
plt.plot(scaler.inverse_transform(close_scaled[-200:]), label="Historical")

# Future predictions
plt.plot(range(200, 200 + future_days), future_output, label="Future Prediction")

plt.legend()
plt.show()
